In [0]:
from databricks.sdk import WorkspaceClient
import pandas as pd

w = WorkspaceClient()

all_runs_data = []

for job in w.jobs.list():

    try:
        runs = w.jobs.list_runs(
            job_id=job.job_id,
            completed_only=True,
            expand_tasks=True
        )

        for run in runs:

            # Single-task job
            if not run.tasks:

                detailed_error = None

                try:
                    if run.state and run.state.result_state:
                        if run.state.result_state.value == "FAILED":
                            output = w.jobs.get_run_output(run.run_id)
                            detailed_error = output.error
                except Exception as e:
                    detailed_error = f"Cannot get output: {str(e)}"

                all_runs_data.append({
                    "job_id": run.job_id,
                    "job_name": run.run_name,
                    "run_id": run.run_id,
                    "task_key": None,
                    "task_run_id": None,
                    "life_cycle_state": run.state.life_cycle_state.value if run.state and run.state.life_cycle_state else None,
                    "result_state": run.state.result_state.value if run.state and run.state.result_state else None,
                    "state_message": run.state.state_message if run.state else None,
                    "error": detailed_error
                })

            # Multi-task job
            else:

                for task in run.tasks:

                    detailed_error = None

                    try:
                        task_output = w.jobs.get_run_output(task.run_id)
                        detailed_error = task_output.error
                    except Exception as e:
                        detailed_error = f"Cannot get output: {str(e)}"

                    all_runs_data.append({
                        "job_id": run.job_id,
                        "job_name": run.run_name,
                        "run_id": run.run_id,
                        "task_key": task.task_key,
                        "task_run_id": task.run_id,
                        "life_cycle_state": run.state.life_cycle_state.value if run.state and run.state.life_cycle_state else None,
                        "result_state": run.state.result_state.value if run.state and run.state.result_state else None,
                        "state_message": run.state.state_message if run.state else None,
                        "error": detailed_error
                    })

    except Exception as e:
        print(f"Failed processing job {job.job_id}: {e}")

df = pd.DataFrame(all_runs_data)

display(df)

job_id,job_name,run_id,task_key,task_run_id,life_cycle_state,result_state,state_message,error
1080231295517612,job1,638792870491425,job1,1035951840160289,INTERNAL_ERROR,FAILED,"Task job1 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-b155a37215d5afb763228829e305ffa1-567978d2af619bda-00]"
1080231295517612,job1,638792870491425,job1,52659750673107,INTERNAL_ERROR,FAILED,"Task job1 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-6e9e20b5d92d09e9a3fdae5d8d92657c-8b0adb9b3f64d435-00]"


In [0]:
spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").saveAsTable("vargabdbx.default.logs")

In [0]:
spark_df = spark.createDataFrame(df)
categorized_df = spark_df.withColumn(
    "analysis",
    expr("""
        ai_query(
            'databricks-qwen3-next-80b-a3b-instruct',
            CONCAT(
                'Analyze this Databricks error and return JSON only. ',
                'Format: ',
                '{"category":"","root_cause":"","recommended_fix":""}. ',
                'Error: ',
                error
            )
        )
    """)
)

display(categorized_df)

job_id,job_name,run_id,task_key,task_run_id,life_cycle_state,result_state,state_message,error,analysis
20297814633820,job1,18915681074355,nb1,23137303296724,INTERNAL_ERROR,FAILED,"Task nb1 failed with message: Workload failed, see run output for details.",NameError: name 'dsfsfs' is not defined,"{ ""category"": ""CodeError"", ""root_cause"": ""The variable or function name 'dsfsfs' is referenced in the code but has not been defined anywhere in the scope."", ""recommended_fix"": ""Check the spelling of 'dsfsfs', ensure it is properly defined before use, or remove the reference if it is unnecessary."" }"
20297814633820,job1,18915681074355,nb1,783238485648827,INTERNAL_ERROR,FAILED,"Task nb1 failed with message: Workload failed, see run output for details.",NameError: name 'dsfsfs' is not defined,"{ ""category"": ""ProgrammingError"", ""root_cause"": ""The variable or function name 'dsfsfs' is referenced in the code but has not been defined anywhere in the scope."", ""recommended_fix"": ""Check the spelling and scope of the variable or function 'dsfsfs'. Define it before use, or correct any typos in its name."" }"


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

all_runs_data = []

for job in w.jobs.list():

    try:
        runs = w.jobs.list_runs(
            job_id=job.job_id,
            completed_only=True,
            expand_tasks=True,
            limit=1  # Latest run only
        )

        for run in runs:

            life_cycle_state = (
                run.state.life_cycle_state.value
                if run.state and run.state.life_cycle_state
                else None
            )

            result_state = (
                run.state.result_state.value
                if run.state and run.state.result_state
                else None
            )

            state_message = (
                run.state.state_message
                if run.state
                else None
            )

            # Skip successful runs
            if result_state != "FAILED":
                continue

            # Normalize single-task and multi-task jobs
            tasks = run.tasks or [None]

            for task in tasks:

                task_run_id = task.run_id if task else run.run_id
                task_key = task.task_key if task else None

                detailed_error = None

                try:
                    output = w.jobs.get_run_output(task_run_id)
                    detailed_error = output.error
                except Exception as e:
                    detailed_error = f"Cannot get output: {str(e)}"

                all_runs_data.append({
                    "job_id": run.job_id,
                    "job_name": run.run_name,
                    "run_id": run.run_id,
                    "task_key": task_key,
                    "task_run_id": task_run_id,
                    "life_cycle_state": life_cycle_state,
                    "result_state": result_state,
                    "state_message": state_message,
                    "error": detailed_error
                })

    except Exception as e:
        print(f"Failed processing job {job.job_id}: {e}")

# Create Spark DataFrame
df = spark.createDataFrame(all_runs_data)

display(df)

error,job_id,job_name,life_cycle_state,result_state,run_id,state_message,task_key,task_run_id
"SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-b155a37215d5afb763228829e305ffa1-567978d2af619bda-00]",1080231295517612,job1,INTERNAL_ERROR,FAILED,638792870491425,"Task job1 failed with message: Workload failed, see run output for details.",job1,1035951840160289
"SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-6e9e20b5d92d09e9a3fdae5d8d92657c-8b0adb9b3f64d435-00]",1080231295517612,job1,INTERNAL_ERROR,FAILED,638792870491425,"Task job1 failed with message: Workload failed, see run output for details.",job1,52659750673107


In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType
)

w = WorkspaceClient()

all_runs_data = []

for job in w.jobs.list():

    try:
        runs = w.jobs.list_runs(
            job_id=job.job_id,
            completed_only=True,
            expand_tasks=True,
            limit=1  # Latest run only
        )

        for run in runs:

            life_cycle_state = None
            result_state = None
            state_message = None
            termination_code = None

            try:
                life_cycle_state = run.state.life_cycle_state.value
            except Exception:
                pass

            try:
                result_state = run.state.result_state.value
            except Exception:
                pass

            try:
                state_message = run.state.state_message
            except Exception:
                pass

            try:
                termination_code = run.status.termination_details.code.value
            except Exception:
                pass

            # Only keep failed runs
            if result_state != "FAILED":
                continue

            # Works for both single-task and multi-task jobs
            tasks = run.tasks or [None]

            for task in tasks:

                task_run_id = task.run_id if task else run.run_id
                task_key = task.task_key if task else None

                detailed_error = None

                try:
                    output = w.jobs.get_run_output(task_run_id)
                    detailed_error = output.error
                except Exception as e:
                    detailed_error = f"Cannot get output: {str(e)}"

                all_runs_data.append({
                    "job_id": run.job_id,
                    "job_name": run.run_name,
                    "run_id": run.run_id,
                    "task_key": task_key,
                    "task_run_id": task_run_id,
                    "life_cycle_state": life_cycle_state,
                    "result_state": result_state,
                    "termination_code": termination_code,
                    "state_message": state_message,
                    "error": detailed_error
                })

    except Exception as e:
        print(f"Failed processing job {job.job_id}: {e}")

# Explicit schema to avoid inference issues
schema = StructType([
    StructField("job_id", LongType(), True),
    StructField("job_name", StringType(), True),
    StructField("run_id", LongType(), True),
    StructField("task_key", StringType(), True),
    StructField("task_run_id", LongType(), True),
    StructField("life_cycle_state", StringType(), True),
    StructField("result_state", StringType(), True),
    StructField("termination_code", StringType(), True),
    StructField("state_message", StringType(), True),
    StructField("error", StringType(), True)
])

df = spark.createDataFrame(all_runs_data, schema=schema)

display(df)

job_id,job_name,run_id,task_key,task_run_id,life_cycle_state,result_state,termination_code,state_message,error
280727186142865,job2,822108505009359,job2,844704539660018,INTERNAL_ERROR,FAILED,RUN_EXECUTION_ERROR,"Task job2 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-d3bdcd8b5bd13bde67dbc015f9b7bd4e-4bb8ac1c137d5c17-00]"
280727186142865,job2,452159087908089,job2,428065314802873,INTERNAL_ERROR,FAILED,RUN_EXECUTION_ERROR,"Task job2 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-80f586993ab1b6c404bb163b3d56934a-7005dde743b92699-00]"


In [0]:
# spark_df = spark.createDataFrame(df)
from pyspark.sql.functions import expr
categorized_df = df.withColumn(
    "analysis",
    expr("""
        ai_query(
            'databricks-qwen3-next-80b-a3b-instruct',
            CONCAT(
                'Analyze this Databricks error and return JSON only. ',
                'Format: ',
                '{"category":"","root_cause":"","recommended_fix":""}. ',
                'Error: ',
                error
            )
        )
    """)
)

display(categorized_df)

job_id,job_name,run_id,task_key,task_run_id,life_cycle_state,result_state,termination_code,state_message,error,analysis
280727186142865,job2,822108505009359,job2,844704539660018,INTERNAL_ERROR,FAILED,RUN_EXECUTION_ERROR,"Task job2 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-d3bdcd8b5bd13bde67dbc015f9b7bd4e-4bb8ac1c137d5c17-00]","{ ""category"": ""SyntaxError"", ""root_cause"": ""The input code snippet is incomplete, likely due to an unclosed block such as a parenthesis, bracket, brace, or multiline statement that was not properly terminated."", ""recommended_fix"": ""Check the command for missing closing delimiters (e.g., ), ], or }), unclosed string literals, or incomplete control structures like if, for, or def. Ensure the entire code block is properly closed and submitted as a complete unit."" }"
280727186142865,job2,452159087908089,job2,428065314802873,INTERNAL_ERROR,FAILED,RUN_EXECUTION_ERROR,"Task job2 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-80f586993ab1b6c404bb163b3d56934a-7005dde743b92699-00]","{ ""category"": ""Syntax Error"", ""root_cause"": ""The input code is incomplete or missing closing brackets, quotes, or colons, causing the parser to expect additional input."", ""recommended_fix"": ""Review the code on line 1 for unclosed parentheses, brackets, braces, or strings, and ensure all statements are properly terminated. If the code spans multiple lines, verify that all lines are correctly concatenated or executed as a complete block."" }"


In [0]:
# spark_df = spark.createDataFrame(df)
from pyspark.sql.functions import expr
categorized_df = df.withColumn(
    "analysis",
    expr("""
        ai_query(
            'databricks-gpt-oss-120b',
            CONCAT(
                'Analyze this Databricks error and return JSON only. ',
                'Format: ',
                '{"category":"","root_cause":"","recommended_fix":""}. ',
                'Error: ',
                error
            )
        )
    """)
)

display(categorized_df)

job_id,job_name,run_id,task_key,task_run_id,life_cycle_state,result_state,termination_code,state_message,error,analysis
280727186142865,job2,822108505009359,job2,844704539660018,INTERNAL_ERROR,FAILED,RUN_EXECUTION_ERROR,"Task job2 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-d3bdcd8b5bd13bde67dbc015f9b7bd4e-4bb8ac1c137d5c17-00]","{ ""category"": ""SyntaxError"", ""root_cause"": ""The submitted command is incomplete—typically due to a missing closing parenthesis, bracket, quote, or an unfinished multi‑line statement—so the Python parser cannot form a complete input."", ""recommended_fix"": ""Inspect the code in the cell and ensure all delimiters (parentheses, brackets, braces, quotes) are properly closed and any block statements (if, for, def, etc.) are completed. Add the missing token(s) and re‑run the cell."" }"
280727186142865,job2,452159087908089,job2,428065314802873,INTERNAL_ERROR,FAILED,RUN_EXECUTION_ERROR,"Task job2 failed with message: Workload failed, see run output for details.","SyntaxError: incomplete input (command-6293009725662879-190104801, line 1) [Trace ID: 00-80f586993ab1b6c404bb163b3d56934a-7005dde743b92699-00]","{""category"":""Syntax Error"",""root_cause"":""The notebook cell contains an incomplete Python statement (e.g., missing closing parenthesis, bracket, quote, or colon), causing the interpreter to reach end‑of‑input without a complete expression."",""recommended_fix"":""Review the code on line 1 of the cell and ensure all statements are complete: close any open parentheses/brackets/quotes, add missing colons after control structures, and finish any multi‑line constructs. Re‑run the cell after correcting the syntax.""}"


In [0]:
run_list = list(w.jobs.list_runs(job_id=280727186142865))[0]
print(run_list)

BaseRun(attempt_number=None, cleanup_duration=0, cluster_instance=None, cluster_spec=None, creator_user_name='vargabghosh@gmail.com', description=None, effective_performance_target=<PerformanceTarget.PERFORMANCE_OPTIMIZED: 'PERFORMANCE_OPTIMIZED'>, effective_usage_policy_id=None, end_time=1780196848081, execution_duration=0, git_source=None, has_more=None, job_clusters=[], job_id=280727186142865, job_parameters=[], job_run_id=822108505009359, number_in_job=822108505009359, original_attempt_run_id=822108505009359, overriding_parameters=None, queue_duration=None, repair_history=[], run_duration=22885, run_id=822108505009359, run_name='job2', run_page_url='https://adb-7405605598355561.1.azuredatabricks.net/?o=7405605598355561#job/280727186142865/run/822108505009359', run_type=<RunType.JOB_RUN: 'JOB_RUN'>, schedule=None, setup_duration=0, start_time=1780196825196, state=RunState(life_cycle_state=<RunLifeCycleState.INTERNAL_ERROR: 'INTERNAL_ERROR'>, queue_reason=None, result_state=<RunResul